# BODAQS Simple Suspension Metrics - Study Set Scope

This notebook opens the shared simple suspension metrics dashboard against a saved Study Set created by the web app, library manager, or another Library API client.

Study Set groupings are included as grouped scope entities, so the dashboard can compare both individual sessions and named groupings from the Study Set.

## 1. Configure Library, Study Set, And Dashboard Defaults

Set `LIBRARIES_ROOT`, `LIBRARY_ID`, and `TARGET_STUDY_SET_ID` before running the dashboard cell. Leave `TARGET_STUDY_SET_ID` blank to list available Study Sets for the configured library.

In [7]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd


def find_analysis_dir(start: Path | None = None) -> Path:
    """Find the analysis package root whether Jupyter starts in repo root or analysis/."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path(r"C:\Users\benco\OneDrive\BODAQS-data")
LIBRARY_ID = "archie"
TARGET_STUDY_SET_ID = "archie-evedon-26-v2"  # Display name "Archie-Evedon-26_v2" also resolves.

# Include Study Set groupings as grouped scope entities. Sessions that are not
# in a grouping remain available as individual session entities.
INCLUDE_STUDY_SET_GROUPINGS = True

# Dashboard parameters: keep these aligned with the preprocessing event schema
# and the one-step suspension metrics workflow used for these sessions.
FRONT_SUSPENSION_SELECTOR = {"end": "front", "domain": "wheel"}
REAR_SUSPENSION_SELECTOR = {"end": "rear", "domain": "wheel"}
FRONT_EVENT_SIGNAL_SELECTOR = {"end": "front", "domain": "wheel"}
REAR_EVENT_SIGNAL_SELECTOR = {"end": "rear", "domain": "wheel"}

SCATTER_COMPRESSION_EVENT_ID = "compressions_all"
SCATTER_REBOUND_EVENT_ID = "rebounds_all"
SCATTER_X_METRIC = "m_stroke_disp_max"
SCATTER_COMPRESSION_Y_METRIC = "m_interval_vel_max"
SCATTER_REBOUND_Y_METRIC = "m_interval_vel_min"

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Libraries root: {LIBRARIES_ROOT}")


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Libraries root: C:\Users\benco\OneDrive\BODAQS-data


## 2. Load Study Set Scope

In [8]:
from bodaqs_analysis.library_api import InvalidStudySetError, LibraryAdapter, make_study_set_selector_handle

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

available_study_sets = adapter.list_study_sets(library_id=LIBRARY_ID)
if not TARGET_STUDY_SET_ID.strip():
    if available_study_sets:
        display(pd.DataFrame(available_study_sets))
    else:
        print("No Study Sets found for this library.")
    raise ValueError("Set TARGET_STUDY_SET_ID to a saved Study Set ID, then rerun this cell.")

try:
    study_set_bridge = adapter.study_set_to_selection_snapshot(
        LIBRARY_ID,
        TARGET_STUDY_SET_ID.strip(),
        include_groupings=INCLUDE_STUDY_SET_GROUPINGS,
    )
except InvalidStudySetError as exc:
    message = str(exc)
    if "one-library Study Sets" in message:
        raise RuntimeError(
            "This pilot notebook currently supports one-library Study Sets only. "
            "Open a Study Set whose sessions all belong to LIBRARY_ID, or wait for the multi-library notebook bridge."
        ) from exc
    raise

sel = make_study_set_selector_handle(
    study_set_bridge,
    title="Sessions/groupings to chart",
    rows=10,
    select_first_by_default=True,
)
key_to_ref = sel["get_key_to_ref"]()
events_index_df = sel["get_events_index_df"]()
entity_snapshot = sel["get_entity_snapshot"]()

print(f"Loaded Study Set: {study_set_bridge['display_name']} ({study_set_bridge['study_set_id']})")
print(f"Sessions selected for charting: {len(key_to_ref)}")
print(f"Scope entities selected for charting: {len(entity_snapshot.selected_entities)}")
display(sel["ui"])
display(events_index_df)


Loaded Study Set: Archie-Evedon-26_v2 (archie-evedon-26-v2)
Sessions selected for charting: 4
Scope entities selected for charting: 1


,session_key,run_id,session_id
4,archie-mega-local_260617_112959::260613_114851,archie-mega-local_260617_112959,260613_114851
5,archie-mega-local_260617_112959::260613_111401,archie-mega-local_260617_112959,260613_111401
6,archie-mega-local_260617_112959::260613_103335,archie-mega-local_260617_112959,260613_103335
7,archie-mega-local_260617_112959::260613_100618,archie-mega-local_260617_112959,260613_100618


## 3. Open Suspension Metrics Dashboard

In [9]:
from bodaqs_analysis.dashboards import make_simple_suspension_metrics_dashboard

if not sel["get_key_to_ref"]():
    raise ValueError("Select at least one Study Set session or grouping before opening the dashboard.")

dashboard = make_simple_suspension_metrics_dashboard(
    sel,
    front_displacement_selector=FRONT_SUSPENSION_SELECTOR,
    rear_displacement_selector=REAR_SUSPENSION_SELECTOR,
    front_velocity_selector=FRONT_SUSPENSION_SELECTOR,
    rear_velocity_selector=REAR_SUSPENSION_SELECTOR,
    front_event_signal_selector=FRONT_EVENT_SIGNAL_SELECTOR,
    rear_event_signal_selector=REAR_EVENT_SIGNAL_SELECTOR,
    compression_event_type=SCATTER_COMPRESSION_EVENT_ID,
    rebound_event_type=SCATTER_REBOUND_EVENT_ID,
    scatter_x_metric=SCATTER_X_METRIC,
    compression_y_metric=SCATTER_COMPRESSION_Y_METRIC,
    rebound_y_metric=SCATTER_REBOUND_Y_METRIC,
)
display(dashboard["ui"])
